In [2]:
import pandas as pd
import numpy as np
from mp_api.client import MPRester
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.core import Structure

# ================= 配置 =================
mp_api_key = '7PYF7c0DvFXufDMo1DL6yLZQbv7ZMpNp'
csv_path = '/home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef_noAO_sg.csv'
output_csv = '/home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_with_FullXRD.csv'

# 读取数据
df = pd.read_csv(csv_path)
if 'material_id' not in df.columns:
    print("正在从 cif_file 列提取 material_id...")
    # 把 .cif 后缀去掉，剩下的就是 ID
    df['material_id'] = df['cif_file'].astype(str).str.replace('.cif', '', regex=False)

# ================= 核心函数：XRD 全谱特征提取 =================
def get_binned_xrd_profile(structure):
    """
    计算原理：运动学衍射理论 (Kinematic Diffraction)
    步骤：
    1. 计算晶面间距 d (基于 Bragg 定律) -> 得到 2Theta 位置
    2. 计算结构因子 F (基于原子散射因子) -> 得到 强度 Intensity
    3. 离散化 (Binning) -> 将谱图映射到固定维度的向量中
    """
    try:
        # 1. 初始化计算器 (模拟铜靶 CuKa 辐射, lambda=1.5406 A)
        calculator = XRDCalculator(wavelength="CuKa")
        
        # 2. 计算谱图 (Pattern)
        # two_theta_range=(0, 90): 只看 0 到 90 度
        pattern = calculator.get_pattern(structure, two_theta_range=(0, 90))
        
        # 3. 离散化 (Binning) - 这就是"方案一"的核心
        # 我们创建 90 个桶，分别代表 0-1度, 1-2度, ..., 89-90度
        # 这种方法就像把连续的波形变成了直方图
        bins = np.zeros(90)
        
        if pattern and len(pattern.x) > 0:
            # 遍历每一个计算出来的峰
            for theta, intensity in zip(pattern.x, pattern.y):
                # 向下取整，找到对应的桶索引
                # 例如 20.8度 -> index 20
                idx = int(theta)
                
                # 边界保护 (防止刚好 90.0 度溢出)
                if 0 <= idx < 90:
                    # 累加强度 (如果有两个峰挤在同一度里，强度叠加)
                    bins[idx] += intensity
            
            # 4. 归一化 (Normalization)
            # 非常重要！因为不同晶体的绝对反射率不同，我们要看的是"相对模式"
            # 将最大值缩放到 1，让特征对数值大小不敏感，只对形状敏感
            max_intensity = np.max(bins)
            if max_intensity > 0:
                bins = bins / max_intensity
                
        return bins.tolist()

    except Exception as e:
        print(f"XRD 计算失败: {e}")
        return [0]*90

# ================= 批量处理 =================
print("正在连接 Materials Project 并计算全谱 XRD 特征...")
mpr = MPRester(mp_api_key)
ids = df['material_id'].tolist()

正在从 cif_file 列提取 material_id...
正在连接 Materials Project 并计算全谱 XRD 特征...


In [3]:
# 批量获取结构
docs = mpr.materials.summary.search(material_ids=ids, fields=["material_id", "structure"])
id_to_struct = {str(d.material_id): d.structure for d in docs}

xrd_features = []
print(f"开始计算 {len(df)} 个结构的 XRD 指纹...")

for i, mid in enumerate(df['material_id']):
    mid = str(mid)
    if mid in id_to_struct:
        feat = get_binned_xrd_profile(id_to_struct[mid])
    else:
        feat = [0]*90
    xrd_features.append(feat)
    
    if i % 100 == 0:
        print(f"进度: {i}/{len(df)}")

# ================= 合并与保存 =================
# 生成列名: XRD_0, XRD_1, ... XRD_89
xrd_col_names = [f'XRD_{i}' for i in range(90)]
df_xrd = pd.DataFrame(xrd_features, columns=xrd_col_names)

# 合并：原始特征 + XRD特征
df_final = pd.concat([df, df_xrd], axis=1)

print("计算完成！")
print(f"新特征维度增加: 90维")
print(f"正在保存到: {output_csv}")
df_final.to_csv(output_csv, index=False)

Retrieving SummaryDoc documents: 100%|██████████| 691/691 [00:00<00:00, 2789474.56it/s]


开始计算 691 个结构的 XRD 指纹...
进度: 0/691
进度: 100/691
进度: 200/691
进度: 300/691
进度: 400/691
进度: 500/691
进度: 600/691
计算完成！
新特征维度增加: 90维
正在保存到: /home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_with_FullXRD.csv
